In [3]:
import pandas as pd
import numpy as np
import json

In [4]:
# with open('note.json', 'r') as file:
#     info = json.loads(file.read())

In [5]:
# info

In [6]:
df = pd.read_parquet('../data/aircraft engine/PM_train.parquet')

In [7]:
df.RUL

0        191
1        190
2        189
3        188
4        187
        ... 
20626      4
20627      3
20628      2
20629      1
20630      0
Name: RUL, Length: 20631, dtype: int64

In [8]:
df['RULC'] = df['RUL'].map(lambda x: 1 if x<30 else 0)

In [9]:
df['RULC']

0        0
1        0
2        0
3        0
4        0
        ..
20626    1
20627    1
20628    1
20629    1
20630    1
Name: RULC, Length: 20631, dtype: int64

In [10]:
df['RULC'].value_counts()[0]/df['RULC'].value_counts()[1]

5.877

In [11]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

In [12]:
from sklearn.linear_model import LogisticRegressionCV

In [13]:
from sklearn.svm import SVC

In [14]:
import xgboost as xgb

In [15]:
from scipy.stats import loguniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, accuracy_score

In [16]:
df.head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s15,s16,s17,s18,s19,s20,s21,max,RUL,RULC
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,191,0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,190,0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,189,0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,188,0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,187,0


In [17]:
x = df.drop(['cycle','max','RUL','RULC'], axis=1)
y = df['RULC']

In [56]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
x_temp, x_val, y_temp, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42)

In [57]:
random_forest = RandomForestClassifier(random_state=42, class_weight={0:1, 1:6})
gradient_boosting = GradientBoostingClassifier(random_state=42 , n_iter_no_change=10)
logistic = LogisticRegressionCV(random_state=42, Cs=11, cv=3, scoring='accuracy', solver='saga', class_weight={0:1, 1:6}, l1_ratios=[0, 0.5, 1])
svc = SVC()
callback = [xgb.callback.EarlyStopping(rounds=10)]
xtreme_gb = xgb.XGBClassifier(random_state=42, scale_pos_weight=6)

In [58]:
rf_p = {
    'n_estimators':np.logspace(4, 5, 5, base=5, dtype=int),
    'criterion':['gini', "entropy", "log_loss"],
    'max_depth':[None, 3, 5, 7, 10],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_features':['sqrt', 'log2'],
    'bootstrap':[True, False],
    'max_samples':[0.8, 1.0],
}

gb_p = {
    'learning_rate':loguniform(1e-4, 100),
    'n_estimators':np.logspace(4, 5, 5, base=5, dtype=int),
    'subsample':[1.0, 0.8, 0.6],
    'criterion':['friedman_mse','squared_error'],
    'max_features':['sqrt', 'log2'],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_depth':[3, 5, 7, 10],
    # tol=0.0001,
    # validation_fraction=0.1,
}

log_p = {
    # 'Cs':11,
    # dual=False, # False if sample>feature
    'penalty':['l2', 'l1', 'elasticnet'],
    # tol=0.0001,
    'max_iter':[100, 200, 300, 500, 1000]

}

svc_p = {
    
}

xgb_p = {
    'objective':['binary:logistic', 'binary:hinge'],
    'n_estimators': np.logspace(4, 5, 5, base=5, dtype=int),               # Set high for early stopping
    'learning_rate': loguniform(1e-4, 100),  # Step size shrinkage
    'max_depth': [None, 3, 5, 7, 10],              # Tree complexity
    'subsample': [0.8, 1.0],             # Rows per tree
    'colsample_bytree': [0.8, 1.0],      # Columns per tree
    'grow_policy': ['depthwise', 'lossguide'],
    # Regularization
    'reg_alpha': [0, 0.1, 1, 5],
    'reg_lambda': [0.1, 1, 10, 20],
    'gamma': [0, 0.1, 1],
    # unbalanced
    # 'min_child_weight': [1, 5, 10],
    # 'eval_metric': ['logloss', 'aucpr']  # Handle class imbalance
}

In [59]:
grid_rf = RandomizedSearchCV(estimator=random_forest, param_distributions=rf_p, cv=3, n_iter=50, random_state=42, scoring='accuracy', verbose=2)
grid_gb = RandomizedSearchCV(estimator=gradient_boosting, param_distributions=gb_p, cv=3, n_iter=50, random_state=42, scoring='accuracy', verbose=2)

In [60]:
grid_log = RandomizedSearchCV(estimator=logistic, param_distributions=log_p, cv=3, n_iter=50, random_state=42, scoring='accuracy', verbose=2)
grid_xgb = RandomizedSearchCV(estimator=xtreme_gb, param_distributions=xgb_p, cv=3, n_iter=50, random_state=42, scoring='accuracy', verbose=2)

In [ ]:
grid_rf.fit(x_train, y_train)
grid_gb.fit(x_train, y_train)

In [ ]:
grid_log.fit(x_train, y_train)

In [61]:
grid_xgb.fit(x_train, y_train)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=depthwise, learning_rate=2.465832945854912, max_depth=10, n_estimators=3125, objective=binary:logistic, reg_alpha=0.1, reg_lambda=10, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=depthwise, learning_rate=2.465832945854912, max_depth=10, n_estimators=3125, objective=binary:logistic, reg_alpha=0.1, reg_lambda=10, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=depthwise, learning_rate=2.465832945854912, max_depth=10, n_estimators=3125, objective=binary:logistic, reg_alpha=0.1, reg_lambda=10, subsample=0.8; total time=   0.8s


/home/harsh/lab/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:46:12] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[CV] END colsample_bytree=0.8, gamma=1, grow_policy=lossguide, learning_rate=0.010051981180656781, max_depth=5, n_estimators=3125, objective=binary:hinge, reg_alpha=5, reg_lambda=20, subsample=1.0; total time=   0.8s


/home/harsh/lab/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:46:13] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[CV] END colsample_bytree=0.8, gamma=1, grow_policy=lossguide, learning_rate=0.010051981180656781, max_depth=5, n_estimators=3125, objective=binary:hinge, reg_alpha=5, reg_lambda=20, subsample=1.0; total time=   0.8s


...

/home/harsh/lab/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:48:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[CV] END colsample_bytree=1.0, gamma=1, grow_policy=lossguide, learning_rate=0.0001419398887474364, max_depth=10, n_estimators=625, objective=binary:hinge, reg_alpha=0, reg_lambda=0.1, subsample=1.0; total time=   3.6s


/home/harsh/lab/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [12:48:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)


[CV] END colsample_bytree=1.0, gamma=1, grow_policy=lossguide, learning_rate=0.0001419398887474364, max_depth=10, n_estimators=625, objective=binary:hinge, reg_alpha=0, reg_lambda=0.1, subsample=1.0; total time=   3.4s


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           gamma=None, grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning_rate...
                                        'grow_policy': ['depthwise',
                                                        'lossguide'],
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fbfbcba64d0>,
                                        'max_depth': [None, 3, 5, 7, 10],
                                        'n_estimators': array([ 625,  934, 1397, 2089, 3125]),
                                        'objective': ['binary:logistic',
                                                      'binary:hinge'],
                                        'reg_alpha': [0, 0.1, 1, 5],
                                        'reg_lambda': [0.1, 1, 10, 20],
                                        'subsample': [0.8, 1.0]},
                   random_state=42, scoring='accuracy', verbose=2)

In [62]:
# y_pred_rf = grid_rf.predict(x_test)
# y_pred_gb = grid_gb.predict(x_test)
# y_pred_log = grid_log.predict(x_test)
y_pred_xgb = grid_xgb.predict(x_test)

In [64]:
try:
    with open('RULC.json', 'r') as file:
        info = json.loads(file.read())
except FileNotFoundError:
    info = {}
print(info)
# info.update({'grid_rf':{'acc':accuracy_score(y_test,y_pred_rf),'best_p':grid_rf.best_params_}})
# # print(info)
# info.update({'grid_gb':{'acc':accuracy_score(y_test,y_pred_gb),'best_p':grid_gb.best_params_}})
# # print(info)
# info.update({'grid_log':{'acc':accuracy_score(y_test,y_pred_log),'best_p':grid_log.best_params_}})
# print(info)
info.update({'grid_xgb_2':{'acc':accuracy_score(y_test,y_pred_xgb),'best_p':grid_xgb.best_params_}})
# print(info)
info

{'grid_rf': {'acc': 0.9651027530050407, 'best_p': {'n_estimators': '934', 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_samples': 0.8, 'max_features': 'sqrt', 'max_depth': None, 'criterion': 'entropy', 'bootstrap': True}}, 'grid_gb': {'acc': 0.9633578906552928, 'best_p': {'criterion': 'squared_error', 'learning_rate': 0.1160068983900715, 'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': '1397', 'subsample': 0.8}}, 'grid_log': {'acc': 0.9024815820085305, 'best_p': {'penalty': 'l1'}}, 'grid_xgb': {'acc': 0.949398991857309, 'best_p': {'colsample_bytree': 0.8, 'gamma': 0.1, 'grow_policy': 'lossguide', 'learning_rate': 0.029256480486087588, 'max_depth': 10, 'n_estimators': '625', 'objective': 'binary:logistic', 'reg_alpha': 1, 'reg_lambda': 1, 'subsample': 1.0}}}


{'grid_rf': {'acc': 0.9651027530050407,
  'best_p': {'n_estimators': '934',
   'min_samples_split': 6,
   'min_samples_leaf': 2,
   'max_samples': 0.8,
   'max_features': 'sqrt',
   'max_depth': None,
   'criterion': 'entropy',
   'bootstrap': True}},
 'grid_gb': {'acc': 0.9633578906552928,
  'best_p': {'criterion': 'squared_error',
   'learning_rate': 0.1160068983900715,
   'max_depth': 3,
   'max_features': 'log2',
   'min_samples_leaf': 5,
   'min_samples_split': 2,
   'n_estimators': '1397',
   'subsample': 0.8}},
 'grid_log': {'acc': 0.9024815820085305, 'best_p': {'penalty': 'l1'}},
 'grid_xgb': {'acc': 0.949398991857309,
  'best_p': {'colsample_bytree': 0.8,
   'gamma': 0.1,
   'grow_policy': 'lossguide',
   'learning_rate': 0.029256480486087588,
   'max_depth': 10,
   'n_estimators': '625',
   'objective': 'binary:logistic',
   'reg_alpha': 1,
   'reg_lambda': 1,
   'subsample': 1.0}},
 'grid_xgb_2': {'acc': 0.9726638231872818,
  'best_p': {'colsample_bytree': 0.8,
   'gamma': 0

In [27]:
type(info)

NameError: name 'info' is not defined

In [65]:
import json
with open('RULC.json', 'w') as file:
    json.dump(info, file, default=str, indent=4)